# Connectivity check

Before anything else: confirm this machine can read and write your own storage container.

You should have saved a `.env` file in the **root of this repository** (the folder that
contains `README.md`). If you haven't, do that now, then run the cells below top to bottom.

This should take about 30 seconds. If it ends in ✅ you're ready for Exercise 1.

In [ ]:
from northtrail import get_spark, path, where_am_i

print(where_am_i())

If the line above says **No .env found**, stop: your `.env` is missing or in the wrong folder.
It belongs next to `README.md`, and it must be named exactly `.env` (not `participant-07.env`).

In [ ]:
# Starting Spark takes ~20 seconds the first time. Later notebooks reuse this same call.
spark = get_spark("sanity-check")
print(f"Spark {spark.version} is up")

In [ ]:
# Write one row to your container, read it back, then clean up.
test_path = path("_sanity", "check")

spark.createDataFrame([(1, "hello NorthTrail")], ["id", "message"]) \
     .write.format("delta").mode("overwrite").save(test_path)

rows = spark.read.format("delta").load(test_path).collect()
print(f"read back: {rows}")

In [ ]:
import shutil

try:
    spark.sql(f"VACUUM delta.`{test_path}` RETAIN 0 HOURS")  # best effort tidy-up
except Exception:
    pass

print()
print("=" * 52)
if rows and rows[0]["message"] == "hello NorthTrail":
    print("✅  Storage is reachable and writable. You're ready.")
else:
    print("❌  Something came back wrong -- call the facilitator.")
print("=" * 52)

## If you saw ❌ or an error

| What you see | What it means |
|---|---|
| `No .env found` | The file isn't in the repo root, or isn't named `.env` |
| `Operation failed: "Server failed to authenticate"` | Your SAS token expired or was copied incompletely — ask for a fresh `.env` |
| `Operation failed: ... 404` | The `CONTAINER` name in your `.env` doesn't exist — check for a typo |
| `Invalid configuration value detected for fs.azure.account.key` | Either your `.env` has no `SAS_TOKEN`, or this kernel already built a Spark session before you saved `.env` | **Restart the kernel** (Kernel → Restart) and run from the top |
| `No FileSystem for scheme "abfss"` | You're not running inside the devcontainer |

Anything else: raise your hand. This is the 15 minutes set aside for exactly this.